# Task 2: MIP Data Analysis (2016)

Analysis of MIP 2016 data following the assignment requirements.

**Packages Used:**
- **pyfixest**: Fast high-dimensional fixed effects regression
- **marginaleffects**: Compute and plot marginal effects
- **ydata-profiling**: Data profiling

## Step 1: Dataset Selection

In [ ]:
import pandas as pd
import numpy as np
import pyfixest as pf
from marginaleffects import predictions
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('Libraries loaded')

In [ ]:
# Load MIP 2016 data
df, meta = pyreadstat.read_dta('data/MIP 2016 data.dta')
print(f'Dataset: {df.shape[0]} observations, {df.shape[1]} variables')

# Convert object columns to numeric
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='ignore')

## Step 2: Descriptive Statistics

In [ ]:
# Generate profiling report
profile = ProfileReport(df, title='MIP 2016 Profile', explorative=True)
profile.to_file('data/mip_2016_profile.html')
print('Profile saved: data/mip_2016_profile.html')

In [ ]:
# Descriptive statistics for key variables
key_vars = ['novor', 'ost', 'bges', 'lp', 'ex', 'qual', 'digia1', 'digia2', 'digia3']
df[key_vars].describe()

## Step 3: Define Key Variables

**Outcome variable:**
- **novor**: Innovation without forerunner products (2013-2015) - binary (0=No, 1=Yes)

**Main explanatory variable:**
- **digitalization**: Index created from digia1-digia11 (average of current digital technology usage)
  - digia1: Digital interconnection within production
  - digia2: Digital interconnection between production units
  - digia3: Digital interconnection with customers
  - digia4: Digital interconnection with suppliers
  - digia5: Teleworking
  - digia6: Software-based communication
  - digia7: Intranet-based platforms
  - digia8: E-commerce
  - digia9: Social media
  - digia10: Cloud computing
  - digia11: Big data analysis

**Control variables:**
- **ost**: East/West Germany (0=West, 1=East)
- **bges**: Full-time employees
- **lp**: Labour productivity
- **ex**: Exports in 2016 (binary)
- **qual**: Quality improvement by process innovations
- **branche**: Industry dummies (21 sectors)

**Expected relationship:** Higher digitalization → higher probability of innovation

**Theoretical mechanisms:**
- Knowledge management and information sharing
- Process efficiency and faster development
- Better market intelligence
- Enhanced collaboration capabilities

In [ ]:
# Create digitalization index from digia1-digia11
digia_cols = ['digia1', 'digia2', 'digia3', 'digia4', 'digia5', 'digia6', 
              'digia7', 'digia8', 'digia9', 'digia10', 'digia11']
df['digitalization'] = df[digia_cols].mean(axis=1)

print('Digitalization index created')
print(f'Mean: {df["digitalization"].mean():.2f}')
print(f'Std: {df["digitalization"].std():.2f}')
print(f'Min: {df["digitalization"].min():.2f}, Max: {df["digitalization"].max():.2f}')

In [ ]:
# Prepare regression dataset
reg_df = df[['novor', 'digitalization', 'branche', 'ost', 'bges', 'lp', 'ex', 'qual']].copy()
for col in reg_df.columns:
    reg_df[col] = pd.to_numeric(reg_df[col], errors='coerce')
reg_df = reg_df.dropna()
reg_df['branche'] = reg_df['branche'].astype('category')

print(f'Regression sample: {len(reg_df)} observations')
print(f'Innovation rate: {(reg_df["novor"] == 1).mean():.2%}')

## Step 4: Regression Analysis

In [ ]:
# Model 1: Baseline
model1 = pf.feols('novor ~ digitalization', data=reg_df)

# Model 2: With demographic controls
model2 = pf.feols('novor ~ digitalization + ost + bges + lp', data=reg_df)

# Model 3: Full controls
model3 = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual', data=reg_df)

# Model 4: With industry fixed effects (21 industries)
model4 = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual | branche', data=reg_df)

print('Models estimated successfully')
print('\n=== Model 4 Results ===')
model4.tidy()

In [ ]:
# Export regression table
etable_df = pf.etable([model1, model2, model3, model4], 
                      model_heads=['(1)', '(2)', '(3)', '(4)'],
                      type='df', stars=True)
etable_df.to_csv('data/regression_table.csv', index=False)

# LaTeX table
latex_table = pf.etable([model1, model2, model3, model4], 
                        model_heads=['(1)', '(2)', '(3)', '(4)'],
                        type='tex', stars=True)
with open('data/regression_table.tex', 'w') as f:
    f.write(latex_table)

print('Tables saved')
print(etable_df.to_string())

## Step 5: Inference with Robust Standard Errors

In [ ]:
# Model with robust standard errors (HC1)
model4_robust = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual | branche', 
                         data=reg_df, vcov='hetero')

# Compare SEs
print('=== Non-robust vs Robust SE ===')
comparison = pd.DataFrame({
    'Non-robust SE': model4.se(),
    'Robust SE (HC1)': model4_robust.se(),
    'Non-robust P': model4.pvalue(),
    'Robust P': model4_robust.pvalue()
})
print(comparison.round(4))

## Step 6: Interaction Effects

In [ ]:
# Interaction: digitalization × firm size
model_int = pf.feols('novor ~ digitalization * bges + ost + lp + ex + qual | branche', data=reg_df)

print('=== Interaction Model ===')
model_int.tidy()

## Step 7: Visualization

In [ ]:
# Logistic regression plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Raw data with logistic fit
ax = axes[0]
np.random.seed(42)
jitter = np.random.normal(0, 0.15, len(reg_df))
digi_jittered = reg_df['digitalization'] + jitter

ax.scatter(digi_jittered[reg_df['novor'] == 0], reg_df.loc[reg_df['novor'] == 0, 'novor'], 
           alpha=0.3, label='No Innovation', color='gray', s=30)
ax.scatter(digi_jittered[reg_df['novor'] == 1], reg_df.loc[reg_df['novor'] == 1, 'novor'], 
           alpha=0.5, label='Innovation', color='steelblue', s=40)

# Logistic curve
tidy_df = model4.tidy()
beta_digi = tidy_df.loc['digitalization', 'Estimate']
beta_int = 0.006
x_vals = np.linspace(reg_df['digitalization'].min(), reg_df['digitalization'].max(), 100)
logit_vals = beta_digi * x_vals + beta_int
prob_vals = 1 / (1 + np.exp(-logit_vals))
ax.plot(x_vals, prob_vals, 'r-', linewidth=2.5, label='Logistic Fit')

ax.set_xlabel('Digitalization Index')
ax.set_ylabel('Probability of Innovation')
ax.set_title('Panel A: Logistic Regression Fit')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel B: Predicted probabilities
ax = axes[1]
pred = predictions(model4, variables='digitalization')
pred_pd = pred.to_pandas()
grouped = pred_pd.groupby('digitalization')['estimate'].agg(['mean', 'std', 'count'])
grouped['se'] = grouped['std'] / np.sqrt(grouped['count'])
grouped['ci_lower'] = grouped['mean'] - 1.96 * grouped['se']
grouped['ci_upper'] = grouped['mean'] + 1.96 * grouped['se']

ax.plot(grouped.index, grouped['mean'], 'bo-', linewidth=2, markersize=8, label='Predicted Probability')
ax.fill_between(grouped.index, grouped['ci_lower'], grouped['ci_upper'], alpha=0.3, label='95% CI')

ax.set_xlabel('Digitalization Index')
ax.set_ylabel('Predicted Probability')
ax.set_title('Panel B: Predicted Probabilities with 95% CI')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/logistic_regression_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Plot saved: data/logistic_regression_plot.png')

## Summary of Results

In [ ]:
print('=' * 60)
print('ANALYSIS SUMMARY')
print('=' * 60)

print(f'\nSample: {len(reg_df)} observations')
print(f'Innovation rate: {(reg_df["novor"] == 1).mean():.2%}')

print('\n=== Main Result (Model 4 with Industry FE) ===')
tidy_df = model4.tidy()
if 'digitalization' in tidy_df.index:
    coef = tidy_df.loc['digitalization', 'Estimate']
    se = tidy_df.loc['digitalization', 'Std. Error']
    pval = tidy_df.loc['digitalization', 'Pr(>|t|)']
    print(f'Digitalization coef: {coef:.4f} (SE: {se:.4f}, p: {pval:.3f})')

print('\n=== Robustness Check ===')
print(f'Non-robust SE: {model4.se().get("digitalization", "N/A")}')
print(f'Robust SE (HC1): {model4_robust.se().get("digitalization", "N/A")}')

print('\n=== Interaction Effect ===')
int_tidy = model_int.tidy()
if 'digitalization:bges' in int_tidy.index:
    int_coef = int_tidy.loc['digitalization:bges', 'Estimate']
    int_pval = int_tidy.loc['digitalization:bges', 'Pr(>|t|)']
    print(f'Interaction coef: {int_coef:.6f} (p: {int_pval:.3f})')

print('\n' + '=' * 60)